# Shakespeare Persona LoRA — Qwen3 14B (Unsloth 4-bit)

Fine-tune Qwen3-14B to speak as **William Shakespeare** across four distinct voice modes drawn from different genres of his plays:

| Mode | Plays | Register |
|---|---|---|
| **Tragic** | Hamlet, Macbeth, King Lear, Othello | Dark introspection, fate, moral collapse |
| **Comic** | Midsummer, Much Ado, Twelfth Night | Wit, wordplay, disguise, romantic chaos |
| **Historical** | Henry V, Richard III, Julius Caesar | Statecraft, power, war rhetoric |
| **Romantic** | Romeo and Juliet | Love poetry, passion, star-crossed fate |

**Stack:** Unsloth + QLoRA 4-bit + TRL SFTTrainer on **DGX Spark (128 GB unified memory)**.

**Deployment target:** A5000 (24 GB) via vLLM with the LoRA adapter mounted on the fly.

**Chat Template:** `<|im_start|>role\ncontent<|im_end|>` (ChatML)

**Architecture:** One unified Shakespeare persona with a single system prompt. The model learns to shift registers (tragic, comic, historical, romantic) based on question content, not separate system prompts.

## How to run
**Press "Run All" and walk away.** Every cell is self-contained and runs in order. No manual cell selection needed.

## 1. Setup — Configuration, Environment, GPU Check

Everything needed before training. Installs missing packages, verifies GPU, sets all paths and hyperparameters. Safe to re-run.

In [1]:
import os, sys, subprocess, importlib, importlib.util

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1: INSTALL MISSING PACKAGES (safe to re-run, never clobbers NGC torch)║
# ║                                                                              ║
# ║  ORDER MATTERS:                                                              ║
# ║    1. torch check (no side effects)                                          ║
# ║    2. pip install small packages                                             ║
# ║    3. fix causal_conv1d (NGC ships broken build w/o CUDA extension)          ║
# ║    4. import unsloth FIRST (BEFORE transformers — required for patching)     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _pip(*args, env_extra=None):
    """Run pip with given args, suppressing output unless it fails."""
    cmd = [sys.executable, "-m", "pip"] + list(args)
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# ── 1a. Verify NGC CUDA PyTorch is intact ──────────────────────────────────────
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "+cpu" in torch.__version__ or "cpu" in torch.__version__:
        print("  NGC CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check Portainer: runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")

print(f"  torch {torch.__version__} — CUDA {torch.version.cuda} — GPU: {torch.cuda.get_device_name(0)}")

# ── 1b. Small utility packages ─────────────────────────────────────────────────
for module, install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(module) is None:
        print(f"  Installing {install_args[-1]}...")
        _pip(*install_args)

# ── 1c. Fix causal_conv1d ──────────────────────────────────────────────────────
#   NGC ships causal_conv1d 1.6.0 Python pkg WITHOUT the compiled CUDA extension
#   (causal_conv1d_cuda). This causes a hard crash when transformers or unsloth
#   try to import FalconH1 model. We must fix this BEFORE importing either.
#
#   CRITICAL: pip caches a broken prebuilt aarch64 wheel. Must use --no-binary
#   to force a source build, plus CAUSAL_CONV1D_FORCE_BUILD=TRUE env var.
#   First build takes ~3 min on aarch64, then cached by pip for future runs.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    # Clean up — don't leave partial imports that spoil unsloth's import order
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing — rebuilding from source (~3 min)...")
    # Uninstall broken version + clear pip cache of broken wheel
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    # Build from source — --no-binary prevents reusing the broken cached wheel
    ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
              "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext — uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed — uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# ── 1d. Import unsloth FIRST, then transformers ────────────────────────────────
#   Unsloth MUST be imported before transformers/trl/peft to apply its
#   monkey-patches. Purge any pre-loaded transformers modules.
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
print(f"  transformers {transformers.__version__}")

# ── 1e. Report final state ─────────────────────────────────────────────────────
print()
for name, mod in [("unsloth", "unsloth"), ("transformers", "transformers"), ("trl", "trl"),
                   ("causal_conv1d", "causal_conv1d")]:
    m = _check_import(mod)
    v = getattr(m, "__version__", "installed") if m else "n/a"
    status = "OK" if m else "FALLBACK" if name == "causal_conv1d" else "MISSING"
    print(f"  {name:25s} {v:20s} [{status}]")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2: CONFIGURATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print()
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

# --- Paths (auto-detect Docker vs host) ---
if os.path.exists("/workspace/training/shakespeare"):
    PROJECT_ROOT = "/workspace/training/shakespeare"
    _env = "Docker (Unsloth container)"
elif os.path.exists("/workspace/shakespeare"):
    PROJECT_ROOT = "/workspace/shakespeare"
    _env = "Docker (legacy mount)"
else:
    PROJECT_ROOT = "/home/spark/projects/training/shakespeare"
    _env = "Host (VS Code / venv)"

OUTPUT_ROOT     = f"{PROJECT_ROOT}/output"

# Input: ShareGPT JSONL from datagen notebook
INPUT_DATA_FILE = f"{PROJECT_ROOT}/data/training-data/shakespeare_persona/shakespeare_sharegpt.jsonl"

# Unified prompt (same as datagen)
PROMPT_FILE     = f"{PROJECT_ROOT}/prompts/shakespeare.md"

# Output: LoRA adapter
BASE_LLM            = "unsloth/Qwen3-14B-unsloth-bnb-4bit"
MODEL_NAME_BASE     = "shakespeare_qwen3_14b_bnb4"
OUTPUT_BASE_DIR     = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR     = f"{OUTPUT_BASE_DIR}/lora_adapters"

# --- Training hyperparameters ---
MAX_SEQ_LENGTH  = 4096
BATCH_SIZE      = 2
GRAD_ACCUM      = 4        # effective batch = 2 × 4 = 8
LEARNING_RATE   = 2e-4
TARGET_EPOCHS   = 1

# --- LoRA ---
LORA_R              = 32
LORA_ALPHA          = 32
LORA_DROPOUT        = 0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# --- Print summary ---
print(f"  Environment:  {_env}")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  Base model:   {BASE_LLM}")
print(f"  LoRA:         r={LORA_R}, alpha={LORA_ALPHA}, targets={len(LORA_TARGET_MODULES)} modules")
print(f"  Training:     batch={BATCH_SIZE} x grad_accum={GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}")
print(f"  Precision:    4-bit QLoRA (pre-quantized NF4)")

# --- Verify paths ---
for path, label in [(INPUT_DATA_FILE, "Training data"), (PROMPT_FILE, "System prompt"), (PROJECT_ROOT, "Project root")]:
    exists = os.path.exists(path)
    print(f"  {'OK' if exists else 'MISSING':7s} {label}: {path}")
    if not exists:
        raise FileNotFoundError(f"{label} not found: {path}")

print()
print("Setup complete. All cells below can run sequentially.")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 — CUDA 13.0 — GPU: NVIDIA GB10
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  transformers 5.2.0

  unsloth                   2026.2.1             [OK]
  transformers              5.2.0                [OK]
  trl                       0.22.2               [OK]
  causal_conv1d             1.6.0                [OK]

CONFIGURATION
  Environment:  Docker (Unsloth container)
  PROJECT_ROOT: /workspace/training/shakespeare
  Base model:   unsloth/Qwen3-14B-unsloth-bnb-4bit
  LoRA:         r=32, alpha=32, targets=7 modules
  Training:     batch=2 x grad_accum=4 = effective 8
  Precision:    4-bit QLoRA (pre-quantized NF4)
  OK      Training data: /workspace/training/shakespeare/data/training-data/shakespeare_persona/shakespeare_sharegpt.jsonl
  OK      System prompt: /workspace/training/shakespeare/prompts

## 2. Load Dataset

Load the ShareGPT JSONL with Shakespeare persona conversations.
- Each conversation has a unified system prompt and alternating human/gpt turns
- Voice modes (tragic, comic, historical, romantic) are stored as metadata

In [2]:
import json, os
from collections import Counter, defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

with open(INPUT_DATA_FILE) as f:
    raw = [json.loads(line) for line in f]

dataset = HFDataset.from_list(raw)

print(f"\n{'='*50}")
print(f"Loaded {len(dataset):,} conversations from {INPUT_DATA_FILE}")
print(f"Columns: {dataset.column_names}")
print(f"Sample keys: {list(raw[0].keys())}")
print(f"First conversation turns: {len(raw[0]['conversations'])}")

LOADING SHAREGPT DATA
  File: /workspace/training/shakespeare/data/training-data/shakespeare_persona/shakespeare_sharegpt.jsonl

Loaded 1,315 conversations from /workspace/training/shakespeare/data/training-data/shakespeare_persona/shakespeare_sharegpt.jsonl
Columns: ['voice_mode', 'conversations']
Sample keys: ['voice_mode', 'conversations']
First conversation turns: 9


## 3. Validate & Summarize

In [3]:
# =========================== VOICE MODE DEFINITIONS ===========================
# Voice modes are for ANALYTICS ONLY — the training data uses ONE unified system prompt.
# The model learns to shift registers based on question content, not system prompt.
VOICE_MODES = {
    "tragic": {
        "description": "Dark introspection — Hamlet, Macbeth, King Lear, Othello",
        "test_prompt": "What drives a man to murder his king and seize the throne?",
    },
    "comic": {
        "description": "Wit and wordplay — Midsummer, Much Ado, Twelfth Night",
        "test_prompt": "Why do lovers make such fools of themselves?",
    },
    "historical": {
        "description": "Statecraft and war — Henry V, Richard III, Julius Caesar",
        "test_prompt": "What does it cost a man to wear the crown?",
    },
    "romantic": {
        "description": "Love poetry, passion, star-crossed fate",
        "test_prompt": "How does a young lover speak when he first beholds his beloved?",
    },
}

# =========================== VALIDATION ===========================
mode_counts = Counter()
total_turns = 0
bad_format  = 0

for conv in raw:
    msgs = conv.get("conversations", [])
    if not msgs or msgs[0].get("from") != "system":
        bad_format += 1
        continue
    # Voice mode stored as metadata field by datagen
    detected = conv.get("voice_mode", "unknown")
    mode_counts[detected] += 1
    total_turns += len(msgs) - 1  # exclude system msg

print(f"VOICE MODE DISTRIBUTION ({len(raw):,} conversations, {total_turns:,} turns):\n")
for mode, count in sorted(mode_counts.items(), key=lambda x: -x[1]):
    pct = count / len(raw) * 100
    desc = VOICE_MODES.get(mode, {}).get("description", "???")
    print(f"  {mode:12s} {count:>5d}  ({pct:>5.1f}%)  {desc}")

if bad_format:
    print(f"\n  ⚠ {bad_format} conversations with bad format (missing system message)")
if mode_counts.get("unknown", 0):
    print(f"  ⚠ {mode_counts['unknown']} conversations with undetected voice mode")

del raw  # Free memory — dataset is now in HF Dataset
print(f"\n✓ Dataset validated")

VOICE MODE DISTRIBUTION (1,315 conversations, 9,858 turns):

  tragic         479  ( 36.4%)  Dark introspection — Hamlet, Macbeth, King Lear, Othello
  comic          359  ( 27.3%)  Wit and wordplay — Midsummer, Much Ado, Twelfth Night
  historical     357  ( 27.1%)  Statecraft and war — Henry V, Richard III, Julius Caesar
  romantic       120  (  9.1%)  Love poetry, passion, star-crossed fate

✓ Dataset validated


## 4. Load Model & Tokenizer (4-bit)

Loads pre-quantized BnB NF4 model directly — no bf16 intermediate step needed.

In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Fix pad token — set pad = eos for causal LM training.
if tokenizer.pad_token is None or tokenizer.pad_token_id != tokenizer.eos_token_id:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"  Set pad_token = eos_token ({tokenizer.eos_token!r}, id={tokenizer.eos_token_id})")

# Align model config so trainer doesn't warn about token mismatch
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

print(f"✓ Model loaded: {BASE_LLM}")
print(f"  Precision: 4-bit QLoRA (pre-quantized NF4)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

  Set pad_token = eos_token ('<|im_end|>', id=151645)
✓ Model loaded: unsloth/Qwen3-14B-unsloth-bnb-4bit
  Precision: 4-bit QLoRA (pre-quantized NF4)
  Max sequence length: 4096
  Vocab size: 151669
  GPU allocated: 11.1 GB


## 5. Format Dataset for Chat Template

Use Unsloth's `standardize_sharegpt` to normalize role names, then apply the Qwen3 tokenizer's native chat template (already ChatML-compatible). Manual sequence packing for 100% token utilization.

In [5]:
# Format dataset for Qwen3 chat template (ChatML) using Unsloth's standardize_sharegpt
from unsloth.chat_templates import standardize_sharegpt
from datasets import Dataset as HFDataset

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
)

# ── Manual sequence packing ──────────────────────────────────────────────────
# Pre-pack: tokenize all conversations → concatenate with EOS separators →
# chunk into fixed MAX_SEQ_LENGTH blocks.  Every training example is a FULL chunk
# with ZERO padding → GPU stays at 100% utilization.

eos_id = tokenizer.eos_token_id
num_conversations = 0
all_ids = []

for text in formatted_texts:
    if not text:
        continue
    ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    all_ids.extend(ids)
    all_ids.append(eos_id)  # EOS separator between conversations
    num_conversations += 1

total_tokens = len(all_ids)
num_chunks = total_tokens // MAX_SEQ_LENGTH
all_ids = all_ids[:num_chunks * MAX_SEQ_LENGTH]  # Discard remainder (< 1 chunk)
chunks = [all_ids[i * MAX_SEQ_LENGTH:(i + 1) * MAX_SEQ_LENGTH] for i in range(num_chunks)]

# Decode back to text — SFTTrainer expects a "text" column
packed_texts = tokenizer.batch_decode(chunks, skip_special_tokens=False)
dataset = HFDataset.from_dict({"text": packed_texts})
dataset = dataset.shuffle(seed=42)

wasted = total_tokens - (num_chunks * MAX_SEQ_LENGTH)
print(f"✓ Dataset packed: {num_conversations} conversations → {num_chunks} chunks of {MAX_SEQ_LENGTH} tokens")
print(f"  Total tokens: {total_tokens:,}  |  Wasted (tail): {wasted:,} ({100*wasted/total_tokens:.1f}%)")
print(f"  Token utilization: ~100% (no padding)")
print(f"\n--- Sample packed text (first 500 chars) ---")
print(dataset[0]["text"][:500])

Unsloth: Standardizing formats (num_proc=24):   0%|          | 0/1315 [00:00<?, ? examples/s]

✓ Dataset packed: 1315 conversations → 1305 chunks of 4096 tokens
  Total tokens: 5,348,009  |  Wasted (tail): 2,729 (0.1%)
  Token utilization: ~100% (no padding)

--- Sample packed text (first 500 chars) ---
 "All the world's a stage"
* Hamnet: My son died in 1596, aged eleven — the plays darkened after
* The Great Tragedies: Hamlet (1600), Othello (1604), King Lear (1605), Macbeth (1606)
* The Romances: The Winter's Tale (1611), The Tempest (1611) — forgiveness after the storm
* Retired: Returned to Stratford, c. 1613 — New Place, the garden, the will
* Died: April 23, 1616 — buried in Holy Trinity Church, Stratford-upon-Avon

### **WORDS I LIVE BY**
* "To be, or not to be, that is the question." (


## 6. Add LoRA Adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing — reduces VRAM, extends context
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✓ LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"✓ Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.2f}%)")
print(f"✓ Target modules: {LORA_TARGET_MODULES}")

Unsloth 2026.2.1 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


✓ LoRA adapters added (r=32, alpha=32)
✓ Trainable: 128,450,560 / 8,691,809,280 params (1.48%)
✓ Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 7. Trainer Setup

In [7]:
from trl import SFTTrainer, SFTConfig

# Data is already pre-packed into MAX_SEQ_LENGTH chunks (zero padding).
# Set packing=False so Unsloth/TRL treats each example as-is.
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,               # ← Already pre-packed, don't re-pack
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        dataset_num_proc=1,           # Unsloth stability
    ),
)

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
num_steps = (len(dataset) * TARGET_EPOCHS + effective_batch_size - 1) // effective_batch_size
print(f"✓ Trainer configured (Qwen3 14B 4-bit QLoRA — pre-packed)")
print(f"  Effective batch size: {BATCH_SIZE} × {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}  |  Steps: ~{num_steps}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: manual (pre-packed, each example = {MAX_SEQ_LENGTH} tokens, zero padding)")
print(f"  Dataset: {len(dataset)} packed chunks")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1305 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✓ Trainer configured (Qwen3 14B 4-bit QLoRA — pre-packed)
  Effective batch size: 2 × 4 = 8
  Epochs: 1  |  Steps: ~164
  LR: 0.0002
  Packing: manual (pre-packed, each example = 4096 tokens, zero padding)
  Dataset: 1305 packed chunks
  Precision: bf16


## 8. Train

In [8]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,305 | Num Epochs = 1 | Total steps = 164
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 128,450,560 of 14,896,757,760 (0.86% trained)


Step,Training Loss
1,2.120089
2,2.177367
3,2.146883
4,2.081679
5,2.071510
6,1.973047
7,1.784637
8,1.730283
9,1.669567
10,1.553410


TrainOutput(global_step=164, training_loss=0.8510098148409914, metrics={'train_runtime': 8154.463, 'train_samples_per_second': 0.16, 'train_steps_per_second': 0.02, 'total_flos': 4.528150929211392e+17, 'train_loss': 0.8510098148409914, 'epoch': 1.0})

## 9. Save LoRA Adapters

Saves adapters + tokenizer + unified system prompt to disk.

In [9]:
import json, shutil
from pathlib import Path

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save unified system prompt alongside adapter for inference use
prompt_save_path = f"{LORA_OUTPUT_DIR}/system_prompt.md"
shutil.copy2(PROMPT_FILE, prompt_save_path)

# Also save as JSON for programmatic access
prompt_json_path = f"{LORA_OUTPUT_DIR}/system_prompt.json"
with open(PROMPT_FILE) as f:
    unified_prompt = f.read().strip()
with open(prompt_json_path, "w") as f:
    json.dump({"shakespeare": unified_prompt}, f, indent=2)

print(f"\n✓ LoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompt:  {prompt_save_path}")
print(f"  System prompt (JSON): {prompt_json_path}")

# List saved files
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

Saving LoRA adapters to /workspace/training/shakespeare/output/shakespeare_qwen3_14b_bnb4/lora_adapters...

✓ LoRA adapters saved!
  Adapters:       /workspace/training/shakespeare/output/shakespeare_qwen3_14b_bnb4/lora_adapters
  System prompt:  /workspace/training/shakespeare/output/shakespeare_qwen3_14b_bnb4/lora_adapters/system_prompt.md
  System prompt (JSON): /workspace/training/shakespeare/output/shakespeare_qwen3_14b_bnb4/lora_adapters/system_prompt.json
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   490.1 MB
  chat_template.jinja                           0.0 MB
  system_prompt.json                            0.0 MB
  system_prompt.md                              0.0 MB
  tokenizer.json                               10.9 MB
  tokenizer_config.json                         0.0 MB


## 10. Test Inference (All Four Voice Modes)

Quick smoke test with the **unified system prompt** — one Shakespeare who shifts registers based on what's being discussed. Each question is designed to naturally elicit a different register.

In [10]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Load unified system prompt
with open(PROMPT_FILE) as f:
    UNIFIED_SYSTEM_PROMPT = f.read().strip()

print(f"✓ Loaded unified prompt ({len(UNIFIED_SYSTEM_PROMPT):,} chars)")
print(f"\nINFERENCE TEST — ONE SHAKESPEARE, FOUR REGISTERS\n")

for mode, info in VOICE_MODES.items():
    question = info["test_prompt"]

    messages = [
        {"role": "system", "content": UNIFIED_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # Shakespeare persona — no thinking tokens
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  REGISTER: {mode.upper()} — {info['description']}")
    print(f"  Q: {question}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

✓ Loaded unified prompt (10,515 chars)

INFERENCE TEST — ONE SHAKESPEARE, FOUR REGISTERS

  REGISTER: TRAGIC — Dark introspection — Hamlet, Macbeth, King Lear, Othello
  Q: What drives a man to murder his king and seize the throne?
  A: 

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 1075, in launch

A crown is not worn by blood alone, but forged in the silence between a man’s ambition and the hour he dares to strike.  

Richard’s hand, that serpent’s hand, was not raised in haste, nor driven by mere hunger—for he knew the weight of murder, the stain it leaves upon the soul and state alike. No, his path was paved with words: the artful lie, the feigned loyalty, the whisper in the ear of power. He did not kill for gold, nor for vengeance, nor even for love—but for dominion. The crown is not a thing to be inherited; it is a prize to be seized, and he who would wear it must first silence those who stand in his way.  

And yet—mark this—the man who murders his king does not always know himself. He speaks of justice, of destiny, of divine right. He plays the part of the avenger even as he plays the part of the villain. Richard doth not merely kill; he *justifies* the killing. He turns regicide into prophecy: “The evil that men do lives after them; the good is oft interred with their bon

## 11. Verify Adapter (Cold Reload)

Loads adapters from disk in a fresh model to confirm portability before deploying to A5000.

In [11]:
# Clean up training model
import gc, torch
del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("✓ Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Reload from disk
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

# Quick test with unified prompt
test_q = "What haunted you when you wrote Hamlet's soliloquy?"

messages = [
    {"role": "system", "content": UNIFIED_SYSTEM_PROMPT},
    {"role": "user", "content": test_q},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
inputs = tokenizer2(text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (unified prompt):")
print(f"  Q: {test_q}")
print(f"  A: {response[:500]}")
print(f"\n✓ Adapter loads cleanly from disk. Ready for deployment via vLLM.")

# List adapter files
print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

✓ Cleared training model from memory
  Loading adapter from: /workspace/training/shakespeare/output/shakespeare_qwen3_14b_bnb4/lora_adapters
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]


ADAPTER RELOAD TEST (unified prompt):
  Q: What haunted you when you wrote Hamlet's soliloquy?
  A: A man may stand upon the battlements of midnight and ask himself whether the world is mad, or only he who sees it so.  

When I shaped that soliloquy, I did not summon a ghost from the grave, but gave voice to the silence that follows when a prince walks alone in a castle that echoes with his thoughts. There is no music in that chamber, no revelry, no whisper of courtiers—only the cold breath of Denmark’s air and the fever in Hamlet’s breast. He stands at the threshold of reason and madness, as 

✓ Adapter loads cleanly from disk. Ready for deployment via vLLM.

Adapter contents:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   490.1 MB
  chat_template.jinja                           0.0 MB
  system_prompt.json                            0.0 MB
  system_prompt.md                   